# CNN-2D — Fase 1 (CV estándar, sin OOF)
**Modelo:** CNN-2D propia (sin pesos preentrenados)  
**Filtros:** Log-Mel · CQT · PCEN · Gammatone  
**Duraciones:** 2s · 3s · 4s · 10s  
**Validación:** Repeated Stratified 5-Fold × 5 repeticiones = **25 folds**  
**Threshold:** dinámico por fold — barrido F1-macro sobre val interna (15% del train)  
**Reporte:** media ± std de métricas por fold (sin acumulación OOF)  

---
Correr las celdas **en orden**. El loop principal puede interrumpirse y reanudarse gracias al checkpoint JSON.

In [1]:
# ─── Celda 1: Imports y configuración global ───────────────────────────────
import gc
import re
import json
import time
import csv
import numpy as np
import matplotlib


matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc, roc_auc_score
)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════════════
CONDITION    = "Depresión"
SPECS_ROOT   = Path("Clasificación Final/Depresión/specs")
RESULTS_ROOT = Path("Depresion/CNN2D_fase1")

FILTER_CONFIGS = [
    {"name": "log-mel",   "img_h": 128},
    {"name": "cqt",       "img_h": 168},
    {"name": "pcen",      "img_h": 128},
    {"name": "gammatone", "img_h": 128},
]

DURATIONS  = [10, 2, 3, 4]
N_SPLITS   = 5
N_REPEATS  = 5
BATCH_SIZE = 128
EPOCHS     = 30
PATIENCE   = 5
VAL_FRAC   = 0.15
THR_GRID   = np.arange(0.05, 0.96, 0.02)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Configuracion cargada")

Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080
Configuracion cargada


In [2]:
# ─── Celda 2: Dataset ──────────────────────────────────────────────────────

def build_file_index(root: Path):
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    stems  = sorted(img_map.keys())
    labels = np.array([stem_label[s] for s in stems], dtype=np.int64)
    return stems, labels, img_map


def make_transform_train(img_h, img_w=None):
    # img_w se ignora: usamos el ancho original de la imagen
    return T.Compose([
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def make_transform_val(img_h, img_w=None):
    return T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


class SpectrogramDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = []
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), torch.tensor(label, dtype=torch.long)


class StemDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = []
        for stem in stems:
            for p in img_map[stem]:
                self.samples.append((p, stem))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), stem

print("Clases de dataset definidas")

Clases de dataset definidas


In [3]:
# ─── Celda 3: Arquitectura CNN-2D ──────────────────────────────────────────

class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_ch, out_ch, n_convs):
            layers = []
            for i in range(n_convs):
                layers += [
                    nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                ]
            return nn.Sequential(*layers, nn.MaxPool2d(2), nn.Dropout2d(0.2))

        self.features = nn.Sequential(
            conv_block(3,   32,  n_convs=2),
            conv_block(32,  64,  n_convs=2),
            conv_block(64,  128, n_convs=3),
            conv_block(128, 256, n_convs=3),
            conv_block(256, 256, n_convs=3),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128,  64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64,    1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x))).squeeze(1)


n_params = sum(p.numel() for p in CNN2D().parameters() if p.requires_grad)
print(f"CNN2D definida — parametros entrenables: {n_params:,}")

CNN2D definida — parametros entrenables: 3,725,601


In [4]:
# ─── Celda 4: Entrenamiento, inferencia y calibración de threshold ─────────
def get_image_width(dur):
    return int(dur * 16000 / 512) + 1

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds    = model(xb).view(-1)
        sample_w = cw_tensor[yb.long()]
        loss     = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def predict_stems(model, stems, img_map, transform):
    """Segment voting: promedio de probabilidades por audio."""
    model.eval()
    ds = StemDataset(stems, img_map, transform)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)
    probs_by_stem = {s: [] for s in stems}
    with torch.no_grad():
        for xb, stem_names in dl:
            xb  = xb.to(DEVICE, non_blocking=True)
            out = model(xb).view(-1).cpu().numpy()
            for prob, stem in zip(out, stem_names):
                probs_by_stem[stem].append(float(prob))
    return {stem: float(np.mean(probs)) for stem, probs in probs_by_stem.items()}


def calibrate_threshold(model, val_stems, val_labels, img_map, transform):
    """Barrido THR_GRID -> threshold que maximiza F1-macro en val interna."""
    val_probs = predict_stems(model, val_stems, img_map, transform)
    probs_arr = np.array([val_probs[s] for s in val_stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        preds = (probs_arr > thr).astype(int)
        score = f1_score(val_labels, preds, average='macro', zero_division=0)
        if score > best_f1:
            best_f1  = score
            best_thr = float(thr)
    return best_thr

print("Funciones de entrenamiento e inferencia definidas")

Funciones de entrenamiento e inferencia definidas


In [5]:
# ─── Celda 5: Checkpoint ───────────────────────────────────────────────────

def ckpt_path(out_dir: Path) -> Path:
    return out_dir / "_checkpoint.json"

def load_checkpoint(out_dir: Path):
    p = ckpt_path(out_dir)
    if p.exists():
        with open(p) as f:
            data = json.load(f)
        print(f"  [CKPT] Reanudando desde fold {data['folds_done']}/25")
        return data
    # fold_metrics ahora incluye 'auc' y 'cm' por fold — sin global acumulado
    return {"folds_done": 0, "fold_metrics": []}

def save_checkpoint(out_dir: Path, ckpt: dict):
    with open(ckpt_path(out_dir), "w") as f:
        json.dump(ckpt, f)

print("Funciones de checkpoint definidas")

Funciones de checkpoint definidas


In [6]:
# ─── Celda 6: Reporte y figuras finales (CV estándar) ──────────────────────
#
# Sin OOF: cada fold aporta sus propias métricas.
# Reporte final = media ± std sobre los 25 folds.
# Figuras:
#   1. Boxplot de métricas
#   2. Distribución de thresholds
#   3. Confusion matrix promediada sobre folds
#   4. Curvas ROC superpuestas (una por fold) + media

def save_results(out_dir, filter_name, dur, ckpt, n_splits, n_repeats):
    fold_metrics = ckpt["fold_metrics"]
    total_folds  = n_splits * n_repeats
    model_label  = f"CNN-2D [{filter_name}] {dur}s"
    fname        = f"cnn2d_{filter_name}_{dur}s"

    keys = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
    metrics_arr = {k: np.array([d[k] for d in fold_metrics]) for k in keys}
    thresholds  = np.array([d["threshold"] for d in fold_metrics])

    # ── Tabla por fold ────────────────────────────────────────────────────────
    print(f"\n{'='*76}")
    print(f"RESULTADOS — {model_label}")
    print(f"{n_splits}-Fold x {n_repeats} repeticiones = {total_folds} folds")
    print(f"{'='*76}")
    print(f"\n  {'Rep-Fold':10s}  {'Thr':5s}  Acc    Prec   Rec    F1     F1-Mac AUC")
    for d in fold_metrics:
        print(f"  R{d['rep']:02d}-F{d['fold']:02d}     "
              f"{d['threshold']:.2f}   "
              f"{d['acc']:.3f}  {d['prec']:.3f}  {d['rec']:.3f}  "
              f"{d['f1']:.3f}  {d['f1mac']:.3f}  {d['auc']:.3f}")

    print(f"\n  {'─'*66}")
    print(f"  {'MEDIA':10s}  {thresholds.mean():.2f}   "
          + "  ".join(f"{metrics_arr[k].mean():.3f}" for k in keys))
    print(f"  {'± STD':10s}  {thresholds.std():.2f}   "
          + "  ".join(f"{metrics_arr[k].std():.3f}" for k in keys))

    # ── Figura 1: Boxplot métricas ────────────────────────────────────────────
    labels_m = {"acc": "Accuracy", "prec": "Precision", "rec": "Recall",
                 "f1": "F1 (pos)", "f1mac": "F1-Macro", "auc": "AUC-ROC"}
    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    fig.suptitle(f"{model_label} — {total_folds} folds", fontsize=11)
    for ax, (k, lbl) in zip(axes, labels_m.items()):
        ax.boxplot(metrics_arr[k], patch_artist=True,
                   boxprops=dict(facecolor='lightsteelblue'),
                   medianprops=dict(color='navy', linewidth=2))
        ax.set_title(lbl, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xticks([])
        ax.axhline(metrics_arr[k].mean(), color='red', linestyle='--',
                   linewidth=1, label=f"u={metrics_arr[k].mean():.3f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(out_dir / f"boxplot_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # ── Figura 2: Distribución thresholds ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(thresholds, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(thresholds.mean(), color='red', linestyle='--',
               label=f"u={thresholds.mean():.2f}  sd={thresholds.std():.2f}")
    ax.set_xlabel("Threshold optimo por fold")
    ax.set_ylabel("Frecuencia")
    ax.set_title(f"Thresholds por fold — {model_label}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"thresholds_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # ── Figura 3: Confusion matrix promediada sobre folds ─────────────────────
    # Cada fold guardó su propia cm como lista 2×2
    cms = np.array([d["cm"] for d in fold_metrics], dtype=float)
    cm_mean = cms.mean(axis=0)
    cm_std  = cms.std(axis=0)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_mean, annot=False, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
    # Anotar manualmente: valor medio ± std en cada celda
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.5,
                    f"{cm_mean[i,j]:.1f}\n+/-{cm_std[i,j]:.1f}",
                    ha='center', va='center', fontsize=12,
                    color='white' if cm_mean[i,j] > cm_mean.max()/2 else 'black')
    ax.set_title(f"Confusion media sobre {total_folds} folds — {model_label}")
    ax.set_ylabel("Real"); ax.set_xlabel("Prediccion")
    plt.tight_layout()
    plt.savefig(out_dir / f"confusion_{fname}.png", dpi=150)
    plt.show()
    plt.close()
    
    # Guardado de matrices de confusion por fold
    cm_dir = out_dir / "confusion_matrices"
    cm_dir.mkdir(exist_ok=True)
    for d in fold_metrics:
        rep = d["rep"]
        fold = d["fold"]
        cm   = np.array(d["cm"])
        #Guardamos csv
        csv_cm_path = cm_dir / f"cm_rep{rep:02d}_fold{fold:02d}.csv"
        np.savetxt(csv_cm_path, cm, delimiter=",", fmt="%d")
        #Guardamos imagen
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                    yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
        plt.title(f"Confusion Rep {rep:02d} Fold {fold:02d} — {model_label}")
        plt.ylabel("Real"); plt.xlabel("Prediccion")
        plt.tight_layout()
        plt.savefig(cm_dir / f"cm_rep{rep:02d}_fold{fold:02d}.png", dpi=150)
        plt.close()
    print(f"   Matrices de confusion por fold guardadas en: {cm_dir}")
    
    # Guardado de curvas ROC por fold
    roc_dir = out_dir / "roc_por_fold"
    roc_dir.mkdir(exist_ok=True)
    for d in fold_metrics:
        fpr = np.array(d["roc_fpr"])
        tpr = np.array(d["roc_tpr"])
        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, lw=2, label=f"AUC = {d['auc']:.3f}")
        plt.plot([0,1], [0,1], 'k--')
        plt.title(f"ROC - Rep {d['rep']:02d} Fold {d['fold']:02d}")
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig(roc_dir / f"roc_R{d['rep']:02d}_F{d['fold']:02d}.png", dpi=150)
        plt.close()

    # ── Figura 4: Curvas ROC por fold superpuestas ────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 7))
    aucs = []
    for d in fold_metrics:
        fpr = np.array(d["roc_fpr"])
        tpr = np.array(d["roc_tpr"])
        fold_auc = d["auc"]
        aucs.append(fold_auc)
        ax.plot(fpr, tpr, alpha=0.3, lw=1, color='steelblue')

    # Curva media interpolada
    mean_fpr = np.linspace(0, 1, 100)
    tprs_interp = []
    for d in fold_metrics:
        fpr = np.array(d["roc_fpr"])
        tpr = np.array(d["roc_tpr"])
        tprs_interp.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs_interp, axis=0)
    std_tpr  = np.std(tprs_interp, axis=0)

    ax.plot(mean_fpr, mean_tpr, color='navy', lw=2,
            label=f"Media AUC = {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    ax.fill_between(mean_fpr,
                    mean_tpr - std_tpr,
                    mean_tpr + std_tpr,
                    alpha=0.15, color='navy', label="+/- 1 std")
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title(f"ROC por fold — {model_label}")
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(out_dir / f"roc_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # ── CSV métricas ──────────────────────────────────────────────────────────
    csv_path = out_dir / f"metrics_{fname}.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["rep","fold","threshold",
                                                "acc","prec","rec","f1","f1mac","auc"])
        writer.writeheader()
        writer.writerows([{k: d[k] for k in ["rep","fold","threshold",
                                               "acc","prec","rec","f1","f1mac","auc"]}
                           for d in fold_metrics])

    print(f"  Guardado en : {out_dir}")
    print(f"  CSV metricas: {csv_path}")

print("Funcion save_results definida")

Funcion save_results definida


In [7]:
# ─── Celda 7: LOOP PRINCIPAL — 4 filtros x 4 duraciones ───────────────────
# Si se interrumpe: volver a correr esta celda, el checkpoint retoma
# desde el ultimo fold completado.

TOTAL_CONFIGS = len(FILTER_CONFIGS) * len(DURATIONS)
config_n = 0

for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    IMG_H       = filt_cfg["img_h"]
    tf_train    = make_transform_train(IMG_H)
    tf_val      = make_transform_val(IMG_H)

    for dur in DURATIONS:
        config_n   += 1
        image_root  = SPECS_ROOT / filter_name / f"{dur}s"
        out_dir     = RESULTS_ROOT / filter_name / f"{dur}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*72}")
        print(f"  CONFIG {config_n}/{TOTAL_CONFIGS} — "
              f"filtro={filter_name}  dur={dur}s  IMG=({IMG_H}x{get_image_width(dur)})")
        print(f"{'='*72}")

        if not image_root.exists():
            print(f"  [SKIP] No existe: {image_root}")
            continue

        file_stems, file_labels, img_map = build_file_index(image_root)
        if len(file_stems) == 0:
            print(f"  [SKIP] Sin imagenes en {image_root}")
            continue

        total_imgs = sum(len(v) for v in img_map.values())
        print(f"\n  Pacientes totales : {len(file_stems)}")
        print(f"  Sin {CONDITION}       : {int((file_labels==0).sum())}")
        print(f"  Con {CONDITION}       : {int((file_labels==1).sum())}")
        print(f"  Segmentos totales : {total_imgs:,} (~{total_imgs/len(file_stems):.1f} por audio)")

        ckpt       = load_checkpoint(out_dir)
        folds_done = ckpt["folds_done"]

        if folds_done >= N_SPLITS * N_REPEATS:
            print(f"\n  [SKIP] Ya completado ({folds_done} folds). Regenerando figuras...")
            save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)
            continue

        rskf = RepeatedStratifiedKFold(
            n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)
        file_stems_arr = np.array(file_stems)
        total_folds    = N_SPLITS * N_REPEATS

        print(f"\n  CV: {N_SPLITS}-Fold x {N_REPEATS} reps = {total_folds} folds")
        if folds_done > 0:
            print(f"  Reanudando desde fold {folds_done + 1}/{total_folds}")

        for fold_idx, (tr_idx, te_idx) in enumerate(
                rskf.split(file_stems_arr, file_labels)):

            if fold_idx < folds_done:
                continue

            repeat_n    = fold_idx // N_SPLITS + 1
            fold_n      = fold_idx %  N_SPLITS + 1
            fold_global = fold_idx + 1

            print(f"\n  Rep {repeat_n}/{N_REPEATS}  |  "
                  f"Fold {fold_n}/{N_SPLITS}  [{fold_global}/{total_folds}]")

            tr_stems  = list(file_stems_arr[tr_idx])
            te_stems  = list(file_stems_arr[te_idx])
            tr_labels = file_labels[tr_idx]
            te_labels = file_labels[te_idx]

            # Val interna 15% para calibrar threshold
            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=VAL_FRAC, random_state=42 + fold_idx)
            tr_sub_idx, val_idx = next(sss.split(tr_stems, tr_labels))

            val_stems       = [tr_stems[i] for i in val_idx]
            val_labels_fold = tr_labels[val_idx]
            train_stems     = [tr_stems[i] for i in tr_sub_idx]
            train_labels    = tr_labels[tr_sub_idx]

            train_ds = SpectrogramDataset(train_stems, img_map, tf_train)
            train_dl = DataLoader(
                train_ds, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=4, pin_memory=True,
                persistent_workers=False,
                prefetch_factor=2,
            )
            print(f"  Train: {len(train_stems)} pac / {len(train_ds):,} segs  "
                  f"| Val: {len(val_stems)} pac  | Test: {len(te_stems)} pac")

            weights   = class_weight.compute_class_weight(
                            'balanced',
                            classes=np.unique(train_labels),
                            y=train_labels)
            cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

            model     = CNN2D().to(DEVICE)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)
            criterion = nn.BCELoss(reduction='none')
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, 'min', factor=0.5, patience=3, min_lr=1e-7)

            best_loss, patience_cnt, best_state = float('inf'), 0, None
            t0 = time.time()

            for epoch in range(EPOCHS):
                loss = train_one_epoch(
                    model, train_dl, optimizer, criterion, cw_tensor)
                scheduler.step(loss)
                if loss < best_loss:
                    best_loss    = loss
                    patience_cnt = 0
                    best_state   = {k: v.cpu().clone()
                                    for k, v in model.state_dict().items()}
                    mark = "v"
                else:
                    patience_cnt += 1
                    mark = f"({patience_cnt}/{PATIENCE})"
                print(f"  Ep {epoch+1:02d}/{EPOCHS}  "
                      f"loss {loss:.4f}  best {best_loss:.4f}  {mark}")
                if patience_cnt >= PATIENCE:
                    print("  Early stopping.")
                    break

            model.load_state_dict(best_state)
            print(f"  Entrenamiento: {time.time()-t0:.1f}s")

            # Calibrar threshold
            best_thr = calibrate_threshold(
                model, val_stems, val_labels_fold, img_map, tf_val)
            print(f"  Threshold optimo: {best_thr:.2f}")

            # Evaluar en test
            stem_probs = predict_stems(model, te_stems, img_map, tf_val)
            fold_true, fold_pred, fold_prob = [], [], []
            for stem, lbl in zip(te_stems, te_labels):
                prob = stem_probs[stem]
                fold_true.append(int(lbl))
                fold_pred.append(1 if prob > best_thr else 0)
                fold_prob.append(prob)

            # ROC y AUC propios del fold
            fold_true_arr = np.array(fold_true)
            fold_prob_arr = np.array(fold_prob)

            # Manejo seguro: si el fold test solo tiene una clase no se puede calcular AUC
            if len(np.unique(fold_true_arr)) > 1:
                fpr, tpr, _ = roc_curve(fold_true_arr, fold_prob_arr)
                fold_auc    = float(auc(fpr, tpr))
                fpr_list    = fpr.tolist()
                tpr_list    = tpr.tolist()
            else:
                fold_auc = 0.0
                fpr = np.array([0.0, 1.0])
                tpr = np.array([0.0, 1.0])
                fpr_list = fpr.tolist()
                tpr_list = tpr.tolist()
                print("  [AVISO] Test de este fold tiene una sola clase — AUC=0")

            # Confusion matrix del fold (como lista para guardar en JSON)
            cm_fold = confusion_matrix(fold_true, fold_pred, labels=[0,1]).tolist()

            m = {
                "rep"      : repeat_n,
                "fold"     : fold_n,
                "threshold": round(best_thr, 4),
                "acc"      : float(accuracy_score(fold_true, fold_pred)),
                "prec"     : float(precision_score(fold_true, fold_pred, zero_division=0)),
                "rec"      : float(recall_score(fold_true, fold_pred, zero_division=0)),
                "f1"       : float(f1_score(fold_true, fold_pred, zero_division=0)),
                "f1mac"    : float(f1_score(fold_true, fold_pred,
                                            average='macro', zero_division=0)),
                "auc"      : fold_auc,
                "cm"       : cm_fold,
                "roc_fpr"  : fpr_list,
                "roc_tpr"  : tpr_list,
            }
            print(f"  acc={m['acc']:.3f}  prec={m['prec']:.3f}  "
                  f"rec={m['rec']:.3f}  F1={m['f1']:.3f}  "
                  f"F1-mac={m['f1mac']:.3f}  AUC={m['auc']:.3f}  "
                  f"thr={m['threshold']:.2f}")

            # Checkpoint
            ckpt["fold_metrics"].append(m)
            ckpt["folds_done"] = fold_global
            save_checkpoint(out_dir, ckpt)

            # Liberar memoria
            del model, train_ds, train_dl, best_state, cw_tensor
            gc.collect()
            torch.cuda.empty_cache()

        save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)

print(f"\n{'='*72}")
print(f"  FASE 1 COMPLETADA — resultados en: {RESULTS_ROOT}")
print(f"{'='*72}")


  CONFIG 1/16 — filtro=log-mel  dur=10s  IMG=(128x313)

  Pacientes totales : 79
  Sin Depresión       : 58
  Con Depresión       : 21
  Segmentos totales : 1,259 (~15.9 por audio)

  CV: 5-Fold x 5 reps = 25 folds

  Rep 1/5  |  Fold 1/5  [1/25]
  Train: 53 pac / 799 segs  | Val: 10 pac  | Test: 16 pac
  Ep 01/30  loss 0.6714  best 0.6714  v
  Ep 02/30  loss 0.6529  best 0.6529  v
  Ep 03/30  loss 0.6455  best 0.6455  v
  Ep 04/30  loss 0.6489  best 0.6455  (1/5)
  Ep 05/30  loss 0.6368  best 0.6368  v
  Ep 06/30  loss 0.6676  best 0.6368  (1/5)
  Ep 07/30  loss 0.6646  best 0.6368  (2/5)
  Ep 08/30  loss 0.6554  best 0.6368  (3/5)
  Ep 09/30  loss 0.6403  best 0.6368  (4/5)
  Ep 10/30  loss 0.6543  best 0.6368  (5/5)
  Early stopping.
  Entrenamiento: 22.6s
  Threshold optimo: 0.49
  acc=0.625  prec=0.429  rec=0.600  F1=0.500  F1-mac=0.600  AUC=0.600  thr=0.49

  Rep 1/5  |  Fold 2/5  [2/25]
  Train: 53 pac / 835 segs  | Val: 10 pac  | Test: 16 pac
  Ep 01/30  loss 0.7168  best 0.71

/tmp/ipykernel_188597/2235653451.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_188597/2235653451.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_188597/2235653451.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


   Matrices de confusion por fold guardadas en: Depresion/CNN2D_fase1/log-mel/10s/confusion_matrices
  Guardado en : Depresion/CNN2D_fase1/log-mel/10s
  CSV metricas: Depresion/CNN2D_fase1/log-mel/10s/metrics_cnn2d_log-mel_10s.csv

  CONFIG 2/16 — filtro=log-mel  dur=2s  IMG=(128x63)

  Pacientes totales : 79
  Sin Depresión       : 58
  Con Depresión       : 21
  Segmentos totales : 12,863 (~162.8 por audio)

  CV: 5-Fold x 5 reps = 25 folds

  Rep 1/5  |  Fold 1/5  [1/25]
  Train: 53 pac / 8,157 segs  | Val: 10 pac  | Test: 16 pac


/tmp/ipykernel_188597/2235653451.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Ep 01/30  loss 0.6595  best 0.6595  v
  Ep 02/30  loss 0.6543  best 0.6543  v
  Ep 03/30  loss 0.6550  best 0.6543  (1/5)
  Ep 04/30  loss 0.6460  best 0.6460  v
  Ep 05/30  loss 0.6404  best 0.6404  v
  Ep 06/30  loss 0.6299  best 0.6299  v
  Ep 07/30  loss 0.6069  best 0.6069  v
  Ep 08/30  loss 0.5752  best 0.5752  v
  Ep 09/30  loss 0.5497  best 0.5497  v
  Ep 10/30  loss 0.5279  best 0.5279  v
  Ep 11/30  loss 0.5176  best 0.5176  v
  Ep 12/30  loss 0.4802  best 0.4802  v
  Ep 13/30  loss 0.4335  best 0.4335  v
  Ep 14/30  loss 0.3859  best 0.3859  v
  Ep 15/30  loss 0.3408  best 0.3408  v
  Ep 16/30  loss 0.2942  best 0.2942  v
  Ep 17/30  loss 0.2738  best 0.2738  v
  Ep 18/30  loss 0.2346  best 0.2346  v
  Ep 19/30  loss 0.2176  best 0.2176  v
  Ep 20/30  loss 0.1837  best 0.1837  v
  Ep 21/30  loss 0.1649  best 0.1649  v
  Ep 22/30  loss 0.1594  best 0.1594  v
  Ep 23/30  loss 0.1503  best 0.1503  v
  Ep 24/30  loss 0.1315  best 0.1315  v
  Ep 25/30  loss 0.1113  best 0.1113

...

/tmp/ipykernel_188597/2235653451.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Ep 01/30  loss 0.6567  best 0.6567  v
  Ep 02/30  loss 0.6560  best 0.6560  v
  Ep 03/30  loss 0.6551  best 0.6551  v
  Ep 04/30  loss 0.6500  best 0.6500  v
  Ep 05/30  loss 0.6495  best 0.6495  v
  Ep 06/30  loss 0.6485  best 0.6485  v
  Ep 07/30  loss 0.6388  best 0.6388  v
  Ep 08/30  loss 0.6385  best 0.6385  v
  Ep 09/30  loss 0.6278  best 0.6278  v
  Ep 10/30  loss 0.6133  best 0.6133  v
  Ep 11/30  loss 0.6055  best 0.6055  v
  Ep 12/30  loss 0.5809  best 0.5809  v
  Ep 13/30  loss 0.5735  best 0.5735  v
  Ep 14/30  loss 0.5387  best 0.5387  v
  Ep 15/30  loss 0.5074  best 0.5074  v
  Ep 16/30  loss 0.4793  best 0.4793  v
  Ep 17/30  loss 0.4693  best 0.4693  v
  Ep 18/30  loss 0.4143  best 0.4143  v
  Ep 19/30  loss 0.4034  best 0.4034  v
  Ep 20/30  loss 0.3832  best 0.3832  v
  Ep 21/30  loss 0.3507  best 0.3507  v
  Ep 22/30  loss 0.3308  best 0.3308  v
  Ep 23/30  loss 0.2956  best 0.2956  v
  Ep 24/30  loss 0.2795  best 0.2795  v
  Ep 25/30  loss 0.2318  best 0.2318  v


/tmp/ipykernel_188597/2235653451.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_188597/2235653451.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_188597/2235653451.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


   Matrices de confusion por fold guardadas en: Depresion/CNN2D_fase1/gammatone/4s/confusion_matrices
  Guardado en : Depresion/CNN2D_fase1/gammatone/4s
  CSV metricas: Depresion/CNN2D_fase1/gammatone/4s/metrics_cnn2d_gammatone_4s.csv

  FASE 1 COMPLETADA — resultados en: Depresion/CNN2D_fase1


/tmp/ipykernel_188597/2235653451.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Para graficar bonitas las graficas de Confusion Matrix y ROC curve
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import auc
from pathlib import Path

def graficar_resultados_bonitos(ckpt, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics = ckpt["fold_metrics"]

    # =================================================================
    # 1. MATRIZ DE CONFUSIÓN GLOBAL (Suma de todos los Folds)
    # =================================================================
    cm_total = np.zeros((2, 2), dtype=int)
    for m in metrics:
        cm_total += np.array(m['cm']) # Sumamos las predicciones

    plt.figure(figsize=(6, 5))
    # Usamos formato 'd' para enteros y un estilo limpio
    sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues', cbar=False,
                annot_kws={"size": 15}, linewidths=1, linecolor='black')

    plt.title("Matriz de Confusión Global (Suma de Folds)", fontsize=14, pad=15)
    plt.ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
    plt.xlabel('Predicción del Modelo', fontsize=12, fontweight='bold')
    plt.xticks([0.5, 1.5], ['0 (Sin Ansiedad)', '1 (Con Ansiedad)'], fontsize=11)
    plt.yticks([0.5, 1.5], ['0 (Sin Ansiedad)', '1 (Con Ansiedad)'], fontsize=11, va='center')

    plt.tight_layout()
    plt.savefig(out_dir / "cm_global_bonita.png", dpi=300, bbox_inches='tight')
    plt.close()

    # =================================================================
    # 2. CURVA ROC PROMEDIO CON DESVIACIÓN ESTÁNDAR SOMBREADA
    # =================================================================
    tprs = []
    aucs = []
    mean_fpr = np.linspace(0, 1, 100) # Base común para poder promediar

    plt.figure(figsize=(8, 6))

    for m in metrics:
        fpr_fold = m['roc_fpr']
        tpr_fold = m['roc_tpr']

        # Interpolar la curva actual al eje común (mean_fpr)
        interp_tpr = np.interp(mean_fpr, fpr_fold, tpr_fold)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(m['auc'])

        # Plotear la línea del fold individual, pero MUY clarita al fondo (opcional)
        plt.plot(fpr_fold, tpr_fold, color='gray', alpha=0.1, lw=1)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)

    # Sombreado de la desviación estándar (+/- 1 std)
    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1) # Asegurar que no pase de 1.0
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0) # Asegurar que no baje de 0.0

    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='#1f77b4', alpha=0.2,
                     label=r'$\pm$ 1 Desv. Estándar')

    # Plot de la línea media
    plt.plot(mean_fpr, mean_tpr, color='#1f77b4', lw=2.5,
             label=f'ROC Promedio (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})')

    # Línea de referencia (Azar)
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='tomato', label='Azar', alpha=0.8)

    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12, fontweight='bold')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12, fontweight='bold')
    plt.title(f'Curva ROC Global con Validación Cruzada', fontsize=14, pad=15)
    plt.legend(loc="lower right", fontsize=11, framealpha=0.9)
    plt.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(out_dir / "roc_global_bonita.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✅ ¡Gráficas generadas exitosamente en {out_dir}!")

<>:72: SyntaxWarning: invalid escape sequence '\p'
<>:72: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_188597/2709666452.py:72: SyntaxWarning: invalid escape sequence '\p'
  label=f'ROC Promedio (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})')


In [9]:
import json
from pathlib import Path

# Ruta de tu checkpoint existente
#ruta_resultados = Path("Depresion/CNN2D_fase1/")
#archivo_json = ruta_resultados / "_checkpoint.json"

# Cargar los datos y graficar
#if archivo_json.exists():
#    with open(archivo_json) as f:
#        ckpt = json.load(f)

#    graficar_resultados_bonitos(ckpt, ruta_resultados)
#else:
#    print("No se encontró el checkpoint.")

✅ ¡Gráficas generadas exitosamente en Ansiedad/CNN2D_fase1/pcen/4s!
